In [1]:
import os
import numpy as np  # linear algebra
import pandas as pd  # data processing, CSV file I/O (e.g. pd.read_csv)



In [2]:
sample_path_candidates = [
    "/kaggle/input/sample_submission.csv",
    "/kaggle/input/stanford-covid-vaccine/sample_submission.csv",
    "../input/sample_submission.csv",
    "../input/stanford-covid-vaccine/sample_submission.csv",
    "/kaggle/data/sample_submission.csv",
    "/kaggle/data/stanford-covid-vaccine/sample_submission.csv",
    "/kaggle/input/stanford-covid-vaccine/sample_submission.csv",
]
sample_path = next((p for p in sample_path_candidates if os.path.exists(p)), None)
if sample_path is None:
    raise FileNotFoundError(
        "Could not find sample_submission.csv in expected locations: "
        + ", ".join(sample_path_candidates)
    )

df = pd.read_csv(sample_path)
df.head()



,id_seqpos,reactivity,deg_Mg_pH10,deg_pH10,deg_Mg_50C,deg_50C
0,id_00b436dec_0,0.0,0.0,0.0,0.0,0.0
1,id_00b436dec_1,0.0,0.0,0.0,0.0,0.0
2,id_00b436dec_2,0.0,0.0,0.0,0.0,0.0
3,id_00b436dec_3,0.0,0.0,0.0,0.0,0.0
4,id_00b436dec_4,0.0,0.0,0.0,0.0,0.0


In [3]:
sequences = list(set(["_".join(v.split("_")[:-1]) for v in df.id_seqpos.values]))
sequences[-10:]



['id_d747b1e6a',
 'id_534207e7c',
 'id_a96fcb4e6',
 'id_0819df711',
 'id_ff103a85c',
 'id_1bb6c57ab',
 'id_1b2d2589a',
 'id_0cae35829',
 'id_55a97fe18',
 'id_cd821844f']

In [4]:
train_path_candidates = [
    "/kaggle/input/train.json",
    "/kaggle/input/stanford-covid-vaccine/train.json",
    "../input/train.json",
    "../input/stanford-covid-vaccine/train.json",
    "/kaggle/data/train.json",
    "/kaggle/data/stanford-covid-vaccine/train.json",
]
train_path = next((p for p in train_path_candidates if os.path.exists(p)), None)
if train_path is None:
    raise FileNotFoundError(
        "Could not find train.json in expected locations: "
        + ", ".join(train_path_candidates)
    )

train = pd.read_json(train_path, lines=True)

target_cols = ["reactivity", "deg_Mg_pH10", "deg_pH10", "deg_Mg_50C", "deg_50C"]
missing_train = [c for c in target_cols if c not in train.columns]
if missing_train:
    raise ValueError(f"train.json is missing required target columns: {missing_train}")

train_use = train.copy()
if "SN_filter" in train_use.columns:
    train_use = train_use.loc[train_use["SN_filter"] == 1].copy()
    if len(train_use) == 0:
        train_use = train.copy()

if "signal_to_noise" in train_use.columns:
    snr_thr = 0.9  # keep as-is to avoid distribution shift risk
    train_snr = train_use.loc[train_use["signal_to_noise"] >= snr_thr].copy()
    if len(train_snr) >= 200:
        train_use = train_snr

seq_scored = int(train_use["seq_scored"].iloc[0])

means_by_pos = {}
global_means = {}

shrink_alpha = 0.958  # keep as-is (existing tuned knob)

error_map = {
    "reactivity": "reactivity_error",
    "deg_Mg_pH10": "deg_error_Mg_pH10",
    "deg_pH10": "deg_error_pH10",
    "deg_Mg_50C": "deg_error_Mg_50C",
    "deg_50C": "deg_error_50C",
}

eps = np.float32(5e-3)  # keep as-is
w_clip = np.float32(1e2)  # keep as-is

# Change (metric-aligned, minimal): cap extremely large reported errors before inverting.
# Rationale: huge errors lead to near-zero weights, effectively discarding those samples;
# clipping errors keeps them as weak-but-non-negligible contributors, often reducing RMSE
# for simple weighted-mean baselines in this competition.
e_clip_max = np.float32(0.35)

for c in target_cols:
    y = np.stack(train_use[c].values).astype(np.float32)  # (n, seq_scored)
    y = y[:, :seq_scored]

    err_col = error_map.get(c, None)
    if err_col is not None and err_col in train_use.columns:
        e = np.stack(train_use[err_col].values).astype(np.float32)
        e = e[:, :seq_scored]

        e = np.clip(e, 0.0, e_clip_max).astype(np.float32)

        w = 1.0 / np.maximum(e, eps)
        w = np.clip(w, 0.0, w_clip).astype(np.float32)
        w = np.where(np.isfinite(w), w, 0.0).astype(np.float32)

        w_sum = w.sum(axis=0)
        safe = w_sum > 0
        pos_mean = np.empty(seq_scored, dtype=np.float32)
        pos_mean[safe] = (w[:, safe] * y[:, safe]).sum(axis=0) / w_sum[safe]
        pos_mean[~safe] = y[:, ~safe].mean(axis=0)

        w_tot = float(w.sum())
        if w_tot > 0:
            glob_mean = float((w * y).sum() / w_tot)
        else:
            glob_mean = float(y.mean())
    else:
        pos_mean = y.mean(axis=0)
        glob_mean = float(y.mean())

    means_by_pos[c] = (
        shrink_alpha * pos_mean + (1.0 - shrink_alpha) * np.float32(glob_mean)
    ).astype(np.float32)
    global_means[c] = np.float32(glob_mean)

seqpos = df["id_seqpos"].str.rsplit("_", n=1).str[-1].astype(int).values
is_scored_pos = seqpos < seq_scored
seqpos_scored = np.clip(seqpos, 0, seq_scored - 1)

for c in target_cols:
    pred = np.empty(len(df), dtype=np.float32)
    pred[is_scored_pos] = means_by_pos[c][seqpos_scored[is_scored_pos]]
    pred[~is_scored_pos] = global_means[c]
    df[c] = pred

df.head()



,id_seqpos,reactivity,deg_Mg_pH10,deg_pH10,deg_Mg_50C,deg_50C
0,id_00b436dec_0,0.533367,0.587425,1.959666,0.477242,0.642366
1,id_00b436dec_1,1.424944,2.037086,2.871142,1.944882,1.805667
2,id_00b436dec_2,1.130200,0.700138,0.905903,0.925189,1.032377
3,id_00b436dec_3,0.830908,0.566242,0.688834,0.778765,0.819717
4,id_00b436dec_4,0.591247,0.643861,0.706088,0.754867,0.710592


In [5]:
target_cols = ["reactivity", "deg_Mg_pH10", "deg_pH10", "deg_Mg_50C", "deg_50C"]
missing = [c for c in target_cols if c not in df.columns]
if missing:
    raise ValueError(f"Submission is missing required columns: {missing}")

for c in target_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0).astype(np.float32)

df[target_cols] /= 1.0002  # keep as-is (existing tuned knob)

df[target_cols] = df[target_cols].replace([np.inf, -np.inf], 0.0).fillna(0.0)

df.head()



,id_seqpos,reactivity,deg_Mg_pH10,deg_pH10,deg_Mg_50C,deg_50C
0,id_00b436dec_0,0.533260,0.587308,1.959274,0.477146,0.642238
1,id_00b436dec_1,1.424659,2.036679,2.870568,1.944493,1.805306
2,id_00b436dec_2,1.129974,0.699998,0.905722,0.925004,1.032171
3,id_00b436dec_3,0.830742,0.566129,0.688697,0.778609,0.819553
4,id_00b436dec_4,0.591129,0.643732,0.705947,0.754716,0.710450


In [6]:
assert df.shape[1] == 6, f"Expected 6 columns, got {df.shape[1]}"
assert (
    list(df.columns) == ["id_seqpos"] + target_cols
), f"Unexpected column order: {df.columns.tolist()}"
assert df["id_seqpos"].isna().sum() == 0, "id_seqpos contains NaNs"
df.describe(include="all").head()



/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,id_seqpos,reactivity,deg_Mg_pH10,deg_pH10,deg_Mg_50C,deg_50C
count,25680,25680.000000,25680.000000,25680.000000,25680.00000,25680.000000
unique,25680,NaN,NaN,NaN,NaN,NaN
top,id_ff103a85c_106,NaN,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,NaN,NaN,NaN
mean,NaN,0.217028,0.332967,0.330262,0.27037,0.305612


In [7]:
df.to_csv("submission.csv", index=False)

print("Wrote submission.csv")
print(df.head())
print("Rows:", len(df), "Cols:", df.shape[1])

Wrote submission.csv
        id_seqpos  reactivity  deg_Mg_pH10  deg_pH10  deg_Mg_50C   deg_50C
0  id_00b436dec_0    0.533260     0.587308  1.959274    0.477146  0.642238
1  id_00b436dec_1    1.424659     2.036679  2.870568    1.944493  1.805306
2  id_00b436dec_2    1.129974     0.699998  0.905722    0.925004  1.032171
3  id_00b436dec_3    0.830742     0.566129  0.688697    0.778609  0.819553
4  id_00b436dec_4    0.591129     0.643732  0.705947    0.754716  0.710450
Rows: 25680 Cols: 6
